# Skin Cancer Classification — Improved Transfer Learning Pipeline
Dataset: ISIC Skin Cancer (9 classes, 5 used) — Kaggle: `nodoubttome/skin-cancer9-classesisic`

**Improvements over the original notebook (applied to every model):**
- Stronger, more varied data augmentation (rotation, shift, shear, zoom, brightness, reflect fill)
- Class-balanced sample weights on every model (not just some)
- Two-phase training: (1) train a deeper, more regularized classification head with the backbone frozen, (2) unfreeze the last ~40% of the backbone and fine-tune at a low learning rate
- Label smoothing + AdamW (decoupled weight decay) instead of plain Adam, for better generalization
- Longer patience / cosine-style LR reduction so models are not stopped too early
- All results (Table 1, Table 2, Table 3) are computed **programmatically** from what the models actually produce when you run this notebook — nothing is hard-coded


In [4]:
!pip -q install kagglehub thop --upgrade

In [5]:
import os, time, random
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report
)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("TensorFlow:", tf.__version__)
gpus = tf.config.list_physical_devices('GPU')
print("GPU available:", bool(gpus), gpus)


TensorFlow: 2.20.0
GPU available: True [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [6]:
# ---- Robust dataset loading: works both inside a Kaggle notebook (dataset attached
# as input) and on Colab/local (downloads via kagglehub). This fixes a real bug from
# before: on Colab, kagglehub needs Kaggle API credentials — if that step silently
# fails, every later cell trains on missing/empty folders, which is exactly what
# produces near-random (~20%, i.e. 1/5 classes) accuracy with no clear error message.

KAGGLE_NATIVE_PATH = "/kaggle/input/skin-cancer9-classesisic"

if os.path.exists(KAGGLE_NATIVE_PATH):
    print("Running inside a Kaggle notebook — using the attached dataset directly.")
    base_path = KAGGLE_NATIVE_PATH
else:
    print("Not on Kaggle — downloading dataset via kagglehub.")
    print("If you get an authentication error here, you need a Kaggle API token:")
    print("  1) kaggle.com -> your profile -> Settings -> Create New API Token (downloads kaggle.json)")
    print("  2) In Colab: upload kaggle.json, then run:")
    print("       import os; os.makedirs('/root/.kaggle', exist_ok=True)")
    print("       !cp kaggle.json /root/.kaggle/ && chmod 600 /root/.kaggle/kaggle.json")
    import kagglehub
    base_path = kagglehub.dataset_download("nodoubttome/skin-cancer9-classesisic")

print("Dataset base path:", base_path)

DATASET_ROOT = os.path.join(base_path, "Skin cancer ISIC The International Skin Imaging Collaboration")
TRAIN_DIR = os.path.join(DATASET_ROOT, "Train")
TEST_DIR = os.path.join(DATASET_ROOT, "Test")

# Fail loudly here instead of silently training on nothing later
assert os.path.exists(TRAIN_DIR), f"Train folder not found at: {TRAIN_DIR}\nCheck the folder structure printed above and fix DATASET_ROOT."
assert os.path.exists(TEST_DIR), f"Test folder not found at: {TEST_DIR}\nCheck the folder structure printed above and fix DATASET_ROOT."

print("Train classes:", os.listdir(TRAIN_DIR))
print("Test classes:", os.listdir(TEST_DIR))


Not on Kaggle — downloading dataset via kagglehub.
If you get an authentication error here, you need a Kaggle API token:
  1) kaggle.com -> your profile -> Settings -> Create New API Token (downloads kaggle.json)
  2) In Colab: upload kaggle.json, then run:
       import os; os.makedirs('/root/.kaggle', exist_ok=True)
       !cp kaggle.json /root/.kaggle/ && chmod 600 /root/.kaggle/kaggle.json
Using Colab cache for faster access to the 'skin-cancer9-classesisic' dataset.
Dataset base path: /kaggle/input/skin-cancer9-classesisic
Train classes: ['pigmented benign keratosis', 'melanoma', 'vascular lesion', 'actinic keratosis', 'squamous cell carcinoma', 'basal cell carcinoma', 'seborrheic keratosis', 'dermatofibroma', 'nevus']
Test classes: ['pigmented benign keratosis', 'melanoma', 'vascular lesion', 'actinic keratosis', 'squamous cell carcinoma', 'basal cell carcinoma', 'seborrheic keratosis', 'dermatofibroma', 'nevus']


In [7]:
# Same 5 classes for every model so comparisons in Table 1/2/3 are fair
SELECTED_CLASSES = [
    "pigmented benign keratosis",
    "melanoma",
    "nevus",
    "basal cell carcinoma",
    "squamous cell carcinoma",
]
NUM_CLASSES = len(SELECTED_CLASSES)
IMG_SIZE = (224, 224)
BATCH_SIZE = 32

IMAGE_EXTENSIONS = (".jpg", ".jpeg", ".png")

def get_image_paths_and_labels(folder, selected_classes):
    image_paths, labels = [], []
    for label, class_name in enumerate(selected_classes):
        class_folder = os.path.join(folder, class_name)
        for file_name in os.listdir(class_folder):
            if file_name.lower().endswith(IMAGE_EXTENSIONS):
                image_paths.append(os.path.join(class_folder, file_name))
                labels.append(label)
    return image_paths, labels

train_paths, train_labels = get_image_paths_and_labels(TRAIN_DIR, SELECTED_CLASSES)
test_paths, test_labels = get_image_paths_and_labels(TEST_DIR, SELECTED_CLASSES)

# Sanity check — if these are 0, everything downstream trains on nothing and gives
# near-random accuracy with no obvious error. Better to stop here with a clear message.
assert len(train_paths) > 0, "No training images found — check SELECTED_CLASSES folder names match TRAIN_DIR exactly."
assert len(test_paths) > 0, "No test images found — check SELECTED_CLASSES folder names match TEST_DIR exactly."

print("Total train+val images:", len(train_paths))
print("Total test images:", len(test_paths))


Total train+val images: 1814
Total test images: 80


In [8]:
# 80/20 stratified split for train/validation
train_paths_split, val_paths, train_labels_split, val_labels = train_test_split(
    train_paths, train_labels,
    test_size=0.20, random_state=SEED, stratify=train_labels
)

print("Training images:", len(train_paths_split))
print("Validation images:", len(val_paths))
print("Testing images:", len(test_paths))

def create_dataframe(image_paths, labels):
    return pd.DataFrame({
        "filename": image_paths,
        "class": [SELECTED_CLASSES[l] for l in labels]
    })

train_df = create_dataframe(train_paths_split, train_labels_split)
val_df = create_dataframe(val_paths, val_labels)
test_df = create_dataframe(test_paths, test_labels)


Training images: 1451
Validation images: 363
Testing images: 80


In [9]:
# ---- IMPROVEMENT 1: class weights applied to EVERY model (original only used them sometimes) ----
class_weights_array = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(train_labels_split),
    y=train_labels_split
)
class_weights = {i: float(w) for i, w in enumerate(class_weights_array)}
print("Class weights:")
for i, w in class_weights.items():
    print(f"  {SELECTED_CLASSES[i]}: {w:.3f}")


Class weights:
  pigmented benign keratosis: 0.786
  melanoma: 0.829
  nevus: 1.015
  basal cell carcinoma: 0.964
  squamous cell carcinoma: 2.001


In [10]:
# ---- IMPROVEMENT 2: one shared, stronger augmentation recipe used by every Keras model ----
from tensorflow.keras.preprocessing.image import ImageDataGenerator

def make_generators(preprocess_input=None, use_rescale=True):
    """
    preprocess_input: a model-specific preprocess_input function (ResNet50/101, DenseNet121).
    use_rescale: if True and preprocess_input is None, applies plain 1/255 rescaling (VGG16/VGG19).
                 If False and preprocess_input is None, images are passed through RAW (0-255).
                 IMPORTANT: EfficientNetB0 has its own internal Rescaling/Normalization layer and
                 expects raw 0-255 input. Applying an extra 1/255 rescale here on top of that
                 (a bug from an earlier version of this notebook) shrinks pixel values to almost
                 zero and can crater accuracy down to ~random-guess level. So EfficientNet must be
                 called with use_rescale=False.
    """
    common_aug = dict(
        rotation_range=30,
        width_shift_range=0.20,
        height_shift_range=0.20,
        shear_range=0.20,
        zoom_range=0.25,
        brightness_range=[0.8, 1.2],
        horizontal_flip=True,
        vertical_flip=False,
        fill_mode="reflect",
    )
    if preprocess_input is not None:
        train_datagen = ImageDataGenerator(preprocessing_function=preprocess_input, **common_aug)
        val_test_datagen = ImageDataGenerator(preprocessing_function=preprocess_input)
    elif use_rescale:
        train_datagen = ImageDataGenerator(rescale=1.0/255.0, **common_aug)
        val_test_datagen = ImageDataGenerator(rescale=1.0/255.0)
    else:
        train_datagen = ImageDataGenerator(**common_aug)
        val_test_datagen = ImageDataGenerator()

    train_gen = train_datagen.flow_from_dataframe(
        train_df, x_col="filename", y_col="class",
        target_size=IMG_SIZE, batch_size=BATCH_SIZE,
        class_mode="categorical", classes=SELECTED_CLASSES,
        shuffle=True, seed=SEED,
    )
    val_gen = val_test_datagen.flow_from_dataframe(
        val_df, x_col="filename", y_col="class",
        target_size=IMG_SIZE, batch_size=BATCH_SIZE,
        class_mode="categorical", classes=SELECTED_CLASSES,
        shuffle=False,
    )
    test_gen = val_test_datagen.flow_from_dataframe(
        test_df, x_col="filename", y_col="class",
        target_size=IMG_SIZE, batch_size=BATCH_SIZE,
        class_mode="categorical", classes=SELECTED_CLASSES,
        shuffle=False,
    )
    return train_gen, val_gen, test_gen


In [32]:
# ---- Faster training: 5 epochs total ----
def train_two_phase(model, base_model, train_gen, val_gen,
                    head_epochs=2, ft_epochs=3, unfreeze_fraction=0.40):

    # ---- Phase 1: train the head only ----
    head_callbacks = [
        keras.callbacks.EarlyStopping(
            monitor="val_loss",
            patience=1,
            restore_best_weights=True,
            verbose=1
        ),
        keras.callbacks.ReduceLROnPlateau(
            monitor="val_loss",
            factor=0.5,
            patience=1,
            min_lr=1e-6,
            verbose=1
        ),
    ]

    history_head = model.fit(
        train_gen,
        validation_data=val_gen,
        epochs=head_epochs,
        class_weight=class_weights,
        callbacks=head_callbacks,
        verbose=1,
    )

    # ---- Phase 2: fine-tuning ----
    base_model.trainable = True

    freeze_until = int(len(base_model.layers) * (1 - unfreeze_fraction))

    for layer in base_model.layers[:freeze_until]:
        layer.trainable = False

    for layer in base_model.layers[freeze_until:]:
        layer.trainable = not isinstance(
            layer, layers.BatchNormalization
        )

    model.compile(
        optimizer=keras.optimizers.AdamW(
            learning_rate=1e-5,
            weight_decay=1e-5
        ),
        loss=keras.losses.CategoricalCrossentropy(
            label_smoothing=0.1
        ),
        metrics=["accuracy"],
    )

    ft_callbacks = [
        keras.callbacks.EarlyStopping(
            monitor="val_loss",
            patience=1,
            restore_best_weights=True,
            verbose=1
        ),
        keras.callbacks.ReduceLROnPlateau(
            monitor="val_loss",
            factor=0.5,
            patience=1,
            min_lr=1e-7,
            verbose=1
        ),
    ]

    history_ft = model.fit(
        train_gen,
        validation_data=val_gen,
        epochs=ft_epochs,
        class_weight=class_weights,
        callbacks=ft_callbacks,
        verbose=1,
    )

    return history_head, history_ft

In [12]:
def evaluate_model(model, test_generator, model_name, is_keras=True):
    test_generator.reset()
    start_time = time.time()
    y_prob = model.predict(test_generator, verbose=1)
    inference_time = time.time() - start_time

    y_pred = np.argmax(y_prob, axis=1)
    y_true = test_generator.classes

    result = {
        "Model": model_name,
        "Accuracy (%)": accuracy_score(y_true, y_pred) * 100,
        "Precision (%)": precision_score(y_true, y_pred, average="macro", zero_division=0) * 100,
        "Recall (%)": recall_score(y_true, y_pred, average="macro", zero_division=0) * 100,
        "F1-Score (%)": f1_score(y_true, y_pred, average="macro", zero_division=0) * 100,
        "AUC (%)": roc_auc_score(y_true, y_prob, multi_class="ovr", average="macro") * 100,
        "Inference Time (s)": inference_time,
        "Params (M)": model.count_params() / 1e6 if is_keras else None,
    }
    print(f"\n{model_name} — test results:")
    for k, v in result.items():
        print(f"  {k}: {v:.4f}" if isinstance(v, float) else f"  {k}: {v}")
    print("\nClassification report:")
    print(classification_report(y_true, y_pred, target_names=SELECTED_CLASSES, digits=4, zero_division=0))
    print("Confusion matrix:")
    print(confusion_matrix(y_true, y_pred))
    return result

# Collected as we go — Table 1 is built from THIS, never hard-coded
all_results = {}


## EfficientNetB0

In [13]:
from tensorflow.keras.applications import EfficientNetB0
preprocess_input = None  # raw 0-255 pixels (EfficientNet has its own internal rescaling)

train_gen_efficientnetb0, val_gen_efficientnetb0, test_gen_efficientnetb0 = make_generators(preprocess_input, use_rescale=False)

# Sanity check: make sure the pixel value range actually matches what this backbone expects.
# For EfficientNet this should print something like min~0, max~255 (raw pixels).
_sample_x, _sample_y = next(train_gen_efficientnetb0)
print("Sample batch pixel range:", _sample_x.min(), "to", _sample_x.max())
print("Sample batch shape:", _sample_x.shape, "| labels shape:", _sample_y.shape)


Found 1451 validated image filenames belonging to 5 classes.
Found 363 validated image filenames belonging to 5 classes.
Found 80 validated image filenames belonging to 5 classes.
Sample batch pixel range: 0.0 to 255.0
Sample batch shape: (32, 224, 224, 3) | labels shape: (32, 5)


In [14]:
efficientnetb0_base = EfficientNetB0(weights="imagenet", include_top=False, input_shape=(224, 224, 3))
efficientnetb0_model = build_transfer_model(efficientnetb0_base, "EfficientNetB0")
efficientnetb0_model.summary()


16705208/16705208 ━━━━━━━━━━━━━━━━━━━━ 2s 0us/step


Model: "EfficientNetB0"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ efficientnetb0 (Functional)     │ (None, 7, 7, 1280)     │     4,049,571 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 1280)           │         5,120 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 1280)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │       327,936 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 256)            │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 128)            │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 5)              │           645 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,417,704 (16.85 MB)

 Trainable params: 364,805 (1.39 MB)

 Non-trainable params: 4,052,899 (15.46 MB)

In [15]:
history_head_efficientnetb0, history_ft_efficientnetb0 = train_two_phase(
    efficientnetb0_model, efficientnetb0_base,
    train_gen_efficientnetb0, val_gen_efficientnetb0,
)


Epoch 1/10
46/46 ━━━━━━━━━━━━━━━━━━━━ 113s 2s/step - accuracy: 0.3239 - loss: 1.8655 - val_accuracy: 0.3333 - val_loss: 1.4525 - learning_rate: 0.0010
Epoch 2/10
46/46 ━━━━━━━━━━━━━━━━━━━━ 38s 821ms/step - accuracy: 0.4425 - loss: 1.5120 - val_accuracy: 0.4160 - val_loss: 1.3775 - learning_rate: 0.0010
Epoch 3/10
46/46 ━━━━━━━━━━━━━━━━━━━━ 38s 819ms/step - accuracy: 0.5024 - loss: 1.3907 - val_accuracy: 0.4876 - val_loss: 1.2923 - learning_rate: 0.0010
Epoch 4/10
46/46 ━━━━━━━━━━━━━━━━━━━━ 37s 814ms/step - accuracy: 0.5231 - loss: 1.3214 - val_accuracy: 0.5152 - val_loss: 1.2659 - learning_rate: 0.0010
Epoch 5/10
46/46 ━━━━━━━━━━━━━━━━━━━━ 37s 805ms/step - accuracy: 0.5431 - loss: 1.2447 - val_accuracy: 0.5482 - val_loss: 1.2345 - learning_rate: 0.0010
Epoch 6/10
46/46 ━━━━━━━━━━━━━━━━━━━━ 37s 798ms/step - accuracy: 0.5693 - loss: 1.2165 - val_accuracy: 0.5537 - val_loss: 1.1977 - learning_rate: 0.0010
Epoch 7/10
46/46 ━━━━━━━━━━━━━━━━━━━━ 37s 810ms/step - accuracy: 0.5824 - loss: 1.17

In [16]:
all_results["EfficientNetB0"] = evaluate_model(
    efficientnetb0_model, test_gen_efficientnetb0, "EfficientNetB0"
)


3/3 ━━━━━━━━━━━━━━━━━━━━ 22s 8s/step

EfficientNetB0 — test results:
  Model: EfficientNetB0
  Accuracy (%): 63.7500
  Precision (%): 64.8065
  Recall (%): 63.7500
  F1-Score (%): 62.8318
  AUC (%): 90.8203
  Inference Time (s): 22.7627
  Params (M): 4.4177

Classification report:
                            precision    recall  f1-score   support

pigmented benign keratosis     0.5263    0.6250    0.5714        16
                  melanoma     0.6000    0.3750    0.4615        16
                     nevus     0.6250    0.9375    0.7500        16
      basal cell carcinoma     0.8462    0.6875    0.7586        16
   squamous cell carcinoma     0.6429    0.5625    0.6000        16

                  accuracy                         0.6375        80
                 macro avg     0.6481    0.6375    0.6283        80
              weighted avg     0.6481    0.6375    0.6283        80

Confusion matrix:
[[10  2  0  1  3]
 [ 1  6  9  0  0]
 [ 1  0 15  0  0]
 [ 2  1  0 11  2]
 [ 5  1  0  1

## VGG16

In [17]:
from tensorflow.keras.applications import VGG16
preprocess_input = None  # simple 1/255 rescaling

train_gen_vgg16, val_gen_vgg16, test_gen_vgg16 = make_generators(preprocess_input, use_rescale=True)

Found 1451 validated image filenames belonging to 5 classes.
Found 363 validated image filenames belonging to 5 classes.
Found 80 validated image filenames belonging to 5 classes.


In [18]:
vgg16_base = VGG16(weights="imagenet", include_top=False, input_shape=(224, 224, 3))
vgg16_model = build_transfer_model(vgg16_base, "VGG16")
vgg16_model.summary()


58889256/58889256 ━━━━━━━━━━━━━━━━━━━━ 4s 0us/step


Model: "VGG16"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ vgg16 (Functional)              │ (None, 7, 7, 512)      │    14,714,688 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_1      │ (None, 512)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 512)            │         2,048 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 256)            │       131,328 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_4           │ (None, 256)            │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_4 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_5           │ (None, 128)            │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_5 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 5)              │           645 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 14,883,141 (56.77 MB)

 Trainable params: 166,661 (651.02 KB)

 Non-trainable params: 14,716,480 (56.14 MB)

In [19]:
history_head_vgg16, history_ft_vgg16 = train_two_phase(
    vgg16_model, vgg16_base,
    train_gen_vgg16, val_gen_vgg16,
)


Epoch 1/10
46/46 ━━━━━━━━━━━━━━━━━━━━ 69s 1s/step - accuracy: 0.3032 - loss: 1.9784 - val_accuracy: 0.3251 - val_loss: 1.5526 - learning_rate: 0.0010
Epoch 2/10
46/46 ━━━━━━━━━━━━━━━━━━━━ 38s 829ms/step - accuracy: 0.3522 - loss: 1.7245 - val_accuracy: 0.3113 - val_loss: 1.5499 - learning_rate: 0.0010
Epoch 3/10
46/46 ━━━━━━━━━━━━━━━━━━━━ 38s 828ms/step - accuracy: 0.4142 - loss: 1.5661 - val_accuracy: 0.3416 - val_loss: 1.5428 - learning_rate: 0.0010
Epoch 4/10
46/46 ━━━━━━━━━━━━━━━━━━━━ 38s 824ms/step - accuracy: 0.4252 - loss: 1.5275 - val_accuracy: 0.2287 - val_loss: 1.5666 - learning_rate: 0.0010
Epoch 5/10
46/46 ━━━━━━━━━━━━━━━━━━━━ 0s 751ms/step - accuracy: 0.4502 - loss: 1.4444
Epoch 5: ReduceLROnPlateau reducing learning rate to 0.0005000000237487257.
46/46 ━━━━━━━━━━━━━━━━━━━━ 39s 857ms/step - accuracy: 0.4459 - loss: 1.4613 - val_accuracy: 0.2259 - val_loss: 1.5504 - learning_rate: 0.0010
Epoch 6/10
46/46 ━━━━━━━━━━━━━━━━━━━━ 40s 877ms/step - accuracy: 0.4480 - loss: 1.4371 

In [20]:
all_results["VGG16"] = evaluate_model(
    vgg16_model, test_gen_vgg16, "VGG16"
)


3/3 ━━━━━━━━━━━━━━━━━━━━ 12s 5s/step

VGG16 — test results:
  Model: VGG16
  Accuracy (%): 52.5000
  Precision (%): 55.6684
  Recall (%): 52.5000
  F1-Score (%): 47.7751
  AUC (%): 84.6289
  Inference Time (s): 12.1731
  Params (M): 14.8831

Classification report:
                            precision    recall  f1-score   support

pigmented benign keratosis     0.4800    0.7500    0.5854        16
                  melanoma     0.6667    0.1250    0.2105        16
                     nevus     0.5000    0.9375    0.6522        16
      basal cell carcinoma     0.6923    0.5625    0.6207        16
   squamous cell carcinoma     0.4444    0.2500    0.3200        16

                  accuracy                         0.5250        80
                 macro avg     0.5567    0.5250    0.4778        80
              weighted avg     0.5567    0.5250    0.4778        80

Confusion matrix:
[[12  0  1  1  2]
 [ 1  2 13  0  0]
 [ 1  0 15  0  0]
 [ 3  1  0  9  3]
 [ 8  0  1  3  4]]


## VGG19

In [21]:
from tensorflow.keras.applications import VGG19
preprocess_input = None  # simple 1/255 rescaling

train_gen_vgg19, val_gen_vgg19, test_gen_vgg19 = make_generators(preprocess_input, use_rescale=True)

Found 1451 validated image filenames belonging to 5 classes.
Found 363 validated image filenames belonging to 5 classes.
Found 80 validated image filenames belonging to 5 classes.


In [22]:
vgg19_base = VGG19(weights="imagenet", include_top=False, input_shape=(224, 224, 3))
vgg19_model = build_transfer_model(vgg19_base, "VGG19")
vgg19_model.summary()


80134624/80134624 ━━━━━━━━━━━━━━━━━━━━ 5s 0us/step


Model: "VGG19"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ vgg19 (Functional)              │ (None, 7, 7, 512)      │    20,024,384 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_2      │ (None, 512)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_6           │ (None, 512)            │         2,048 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_6 (Dropout)             │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 256)            │       131,328 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_7           │ (None, 256)            │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_7 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_8           │ (None, 128)            │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_8 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_8 (Dense)                 │ (None, 5)              │           645 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 20,192,837 (77.03 MB)

 Trainable params: 166,661 (651.02 KB)

 Non-trainable params: 20,026,176 (76.39 MB)

In [26]:
# VGG19 — total 5 epochs
history_head_vgg19, history_ft_vgg19 = train_two_phase(
    vgg19_model,
    vgg19_base,
    train_gen_vgg19,
    val_gen_vgg19,
    head_epochs=2,
    ft_epochs=3
)


Epoch 1/2
46/46 ━━━━━━━━━━━━━━━━━━━━ 43s 912ms/step - accuracy: 0.3212 - loss: 1.7775 - val_accuracy: 0.2342 - val_loss: 1.6480 - learning_rate: 0.0010
Epoch 2/2
46/46 ━━━━━━━━━━━━━━━━━━━━ 40s 861ms/step - accuracy: 0.3473 - loss: 1.6336 - val_accuracy: 0.1102 - val_loss: 1.6345 - learning_rate: 0.0010
Restoring model weights from the end of the best epoch: 2.
Epoch 1/3
46/46 ━━━━━━━━━━━━━━━━━━━━ 59s 1s/step - accuracy: 0.3467 - loss: 1.6208 - val_accuracy: 0.1377 - val_loss: 1.5934 - learning_rate: 1.0000e-05
Epoch 2/3
46/46 ━━━━━━━━━━━━━━━━━━━━ 42s 901ms/step - accuracy: 0.4287 - loss: 1.4901 - val_accuracy: 0.2314 - val_loss: 1.5395 - learning_rate: 1.0000e-05
Epoch 3/3
46/46 ━━━━━━━━━━━━━━━━━━━━ 91s 1s/step - accuracy: 0.4349 - loss: 1.4793 - val_accuracy: 0.3719 - val_loss: 1.5006 - learning_rate: 1.0000e-05
Restoring model weights from the end of the best epoch: 3.


In [27]:
all_results["VGG19"] = evaluate_model(
    vgg19_model, test_gen_vgg19, "VGG19"
)


2/3 ━━━━━━━━━━━━━━━━━━━━ 0s 215ms/step

3/3 ━━━━━━━━━━━━━━━━━━━━ 4s 1s/step

VGG19 — test results:
  Model: VGG19
  Accuracy (%): 46.2500
  Precision (%): 29.2291
  Recall (%): 46.2500
  F1-Score (%): 35.4550
  AUC (%): 78.3594
  Inference Time (s): 4.7897
  Params (M): 20.1928

Classification report:
                            precision    recall  f1-score   support

pigmented benign keratosis     0.0000    0.0000    0.0000        16
                  melanoma     0.4118    0.8750    0.5600        16
                     nevus     0.0000    0.0000    0.0000        16
      basal cell carcinoma     0.4783    0.6875    0.5641        16
   squamous cell carcinoma     0.5714    0.7500    0.6486        16

                  accuracy                         0.4625        80
                 macro avg     0.2923    0.4625    0.3546        80
              weighted avg     0.2923    0.4625    0.3546        80

Confusion matrix:
[[ 0  3  0  8  5]
 [ 0 14  1  0  1]
 [ 0 15  0  1  0]
 [ 1  1  0 11  3]
 [ 0  1  0  3 12]]


## ResNet50

In [28]:
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.applications.resnet50 import preprocess_input

train_gen_resnet50, val_gen_resnet50, test_gen_resnet50 = make_generators(preprocess_input, use_rescale=True)

Found 1451 validated image filenames belonging to 5 classes.
Found 363 validated image filenames belonging to 5 classes.
Found 80 validated image filenames belonging to 5 classes.


In [29]:
resnet50_base = ResNet50(weights="imagenet", include_top=False, input_shape=(224, 224, 3))
resnet50_model = build_transfer_model(resnet50_base, "ResNet50")
resnet50_model.summary()


94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 6s 0us/step


Model: "ResNet50"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ resnet50 (Functional)           │ (None, 7, 7, 2048)     │    23,587,712 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_3      │ (None, 2048)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_9           │ (None, 2048)           │         8,192 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_9 (Dropout)             │ (None, 2048)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_9 (Dense)                 │ (None, 256)            │       524,544 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_10          │ (None, 256)            │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_10 (Dropout)            │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_10 (Dense)                │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_11          │ (None, 128)            │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_11 (Dropout)            │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_11 (Dense)                │ (None, 5)              │           645 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 24,155,525 (92.15 MB)

 Trainable params: 562,949 (2.15 MB)

 Non-trainable params: 23,592,576 (90.00 MB)

In [33]:
history_head_resnet50, history_ft_resnet50 = train_two_phase(
    resnet50_model, resnet50_base,
    train_gen_resnet50, val_gen_resnet50,
)


Epoch 1/2
46/46 ━━━━━━━━━━━━━━━━━━━━ 40s 859ms/step - accuracy: 0.6940 - loss: 1.0054 - val_accuracy: 0.6667 - val_loss: 1.0786 - learning_rate: 5.0000e-04
Epoch 2/2
46/46 ━━━━━━━━━━━━━━━━━━━━ 38s 828ms/step - accuracy: 0.6954 - loss: 1.0132 - val_accuracy: 0.6584 - val_loss: 1.0721 - learning_rate: 5.0000e-04
Restoring model weights from the end of the best epoch: 2.
Epoch 1/3
46/46 ━━━━━━━━━━━━━━━━━━━━ 79s 1s/step - accuracy: 0.7078 - loss: 0.9949 - val_accuracy: 0.6694 - val_loss: 1.0293 - learning_rate: 1.0000e-05
Epoch 2/3
46/46 ━━━━━━━━━━━━━━━━━━━━ 0s 764ms/step - accuracy: 0.7135 - loss: 1.0224
Epoch 2: ReduceLROnPlateau reducing learning rate to 4.999999873689376e-06.
46/46 ━━━━━━━━━━━━━━━━━━━━ 39s 853ms/step - accuracy: 0.7174 - loss: 0.9768 - val_accuracy: 0.6777 - val_loss: 1.0710 - learning_rate: 1.0000e-05
Epoch 2: early stopping
Restoring model weights from the end of the best epoch: 1.


In [34]:
all_results["ResNet50"] = evaluate_model(
    resnet50_model, test_gen_resnet50, "ResNet50"
)


3/3 ━━━━━━━━━━━━━━━━━━━━ 12s 4s/step

ResNet50 — test results:
  Model: ResNet50
  Accuracy (%): 61.2500
  Precision (%): 63.3810
  Recall (%): 61.2500
  F1-Score (%): 60.8723
  AUC (%): 87.2070
  Inference Time (s): 12.1299
  Params (M): 24.1555

Classification report:
                            precision    recall  f1-score   support

pigmented benign keratosis     0.5000    0.6875    0.5789        16
                  melanoma     0.7000    0.4375    0.5385        16
                     nevus     0.6190    0.8125    0.7027        16
      basal cell carcinoma     0.6000    0.5625    0.5806        16
   squamous cell carcinoma     0.7500    0.5625    0.6429        16

                  accuracy                         0.6125        80
                 macro avg     0.6338    0.6125    0.6087        80
              weighted avg     0.6338    0.6125    0.6087        80

Confusion matrix:
[[11  1  0  3  1]
 [ 1  7  8  0  0]
 [ 3  0 13  0  0]
 [ 5  0  0  9  2]
 [ 2  2  0  3  9]]


## ResNet101

In [35]:
from tensorflow.keras.applications import ResNet101
from tensorflow.keras.applications.resnet import preprocess_input

train_gen_resnet101, val_gen_resnet101, test_gen_resnet101 = make_generators(preprocess_input, use_rescale=True)

Found 1451 validated image filenames belonging to 5 classes.
Found 363 validated image filenames belonging to 5 classes.
Found 80 validated image filenames belonging to 5 classes.


In [36]:
resnet101_base = ResNet101(weights="imagenet", include_top=False, input_shape=(224, 224, 3))
resnet101_model = build_transfer_model(resnet101_base, "ResNet101")
resnet101_model.summary()


171446536/171446536 ━━━━━━━━━━━━━━━━━━━━ 9s 0us/step


Model: "ResNet101"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ resnet101 (Functional)          │ (None, 7, 7, 2048)     │    42,658,176 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_4      │ (None, 2048)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_12          │ (None, 2048)           │         8,192 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_12 (Dropout)            │ (None, 2048)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_12 (Dense)                │ (None, 256)            │       524,544 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_13          │ (None, 256)            │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_13 (Dropout)            │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_13 (Dense)                │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_14          │ (None, 128)            │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_14 (Dropout)            │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_14 (Dense)                │ (None, 5)              │           645 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 43,225,989 (164.89 MB)

 Trainable params: 562,949 (2.15 MB)

 Non-trainable params: 42,663,040 (162.75 MB)

In [37]:
history_head_resnet101, history_ft_resnet101 = train_two_phase(
    resnet101_model, resnet101_base,
    train_gen_resnet101, val_gen_resnet101,
)


Epoch 1/2
46/46 ━━━━━━━━━━━━━━━━━━━━ 79s 1s/step - accuracy: 0.3660 - loss: 1.7934 - val_accuracy: 0.4490 - val_loss: 1.4851 - learning_rate: 0.0010
Epoch 2/2
46/46 ━━━━━━━━━━━━━━━━━━━━ 41s 887ms/step - accuracy: 0.5059 - loss: 1.4188 - val_accuracy: 0.5482 - val_loss: 1.2982 - learning_rate: 0.0010
Restoring model weights from the end of the best epoch: 2.
Epoch 1/3
46/46 ━━━━━━━━━━━━━━━━━━━━ 102s 1s/step - accuracy: 0.5637 - loss: 1.2944 - val_accuracy: 0.5592 - val_loss: 1.2600 - learning_rate: 1.0000e-05
Epoch 2/3
46/46 ━━━━━━━━━━━━━━━━━━━━ 0s 832ms/step - accuracy: 0.5671 - loss: 1.2412
Epoch 2: ReduceLROnPlateau reducing learning rate to 4.999999873689376e-06.
46/46 ━━━━━━━━━━━━━━━━━━━━ 43s 930ms/step - accuracy: 0.5658 - loss: 1.2527 - val_accuracy: 0.5482 - val_loss: 1.2688 - learning_rate: 1.0000e-05
Epoch 2: early stopping
Restoring model weights from the end of the best epoch: 1.


In [38]:
all_results["ResNet101"] = evaluate_model(
    resnet101_model, test_gen_resnet101, "ResNet101"
)


3/3 ━━━━━━━━━━━━━━━━━━━━ 16s 5s/step

ResNet101 — test results:
  Model: ResNet101
  Accuracy (%): 48.7500
  Precision (%): 46.7964
  Recall (%): 48.7500
  F1-Score (%): 44.8295
  AUC (%): 81.9336
  Inference Time (s): 16.8073
  Params (M): 43.2260

Classification report:
                            precision    recall  f1-score   support

pigmented benign keratosis     0.5500    0.6875    0.6111        16
                  melanoma     0.2500    0.1875    0.2143        16
                     nevus     0.5517    1.0000    0.7111        16
      basal cell carcinoma     0.5714    0.2500    0.3478        16
   squamous cell carcinoma     0.4167    0.3125    0.3571        16

                  accuracy                         0.4875        80
                 macro avg     0.4680    0.4875    0.4483        80
              weighted avg     0.4680    0.4875    0.4483        80

Confusion matrix:
[[11  2  0  1  2]
 [ 1  3 12  0  0]
 [ 0  0 16  0  0]
 [ 5  2  0  4  5]
 [ 3  5  1  2  5]]


## DenseNet121

In [39]:
from tensorflow.keras.applications import DenseNet121
from tensorflow.keras.applications.densenet import preprocess_input

train_gen_densenet121, val_gen_densenet121, test_gen_densenet121 = make_generators(preprocess_input, use_rescale=True)

Found 1451 validated image filenames belonging to 5 classes.
Found 363 validated image filenames belonging to 5 classes.
Found 80 validated image filenames belonging to 5 classes.


In [40]:
densenet121_base = DenseNet121(weights="imagenet", include_top=False, input_shape=(224, 224, 3))
densenet121_model = build_transfer_model(densenet121_base, "DenseNet121")
densenet121_model.summary()


29084464/29084464 ━━━━━━━━━━━━━━━━━━━━ 2s 0us/step


Model: "DenseNet121"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ densenet121 (Functional)        │ (None, 7, 7, 1024)     │     7,037,504 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_5      │ (None, 1024)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_15          │ (None, 1024)           │         4,096 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_15 (Dropout)            │ (None, 1024)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_15 (Dense)                │ (None, 256)            │       262,400 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_16          │ (None, 256)            │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_16 (Dropout)            │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_16 (Dense)                │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_17          │ (None, 128)            │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_17 (Dropout)            │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_17 (Dense)                │ (None, 5)              │           645 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 7,339,077 (28.00 MB)

 Trainable params: 298,757 (1.14 MB)

 Non-trainable params: 7,040,320 (26.86 MB)

In [41]:
history_head_densenet121, history_ft_densenet121 = train_two_phase(
    densenet121_model, densenet121_base,
    train_gen_densenet121, val_gen_densenet121,
)


Epoch 1/2
46/46 ━━━━━━━━━━━━━━━━━━━━ 107s 2s/step - accuracy: 0.3908 - loss: 1.7274 - val_accuracy: 0.4959 - val_loss: 1.3079 - learning_rate: 0.0010
Epoch 2/2
46/46 ━━━━━━━━━━━━━━━━━━━━ 38s 833ms/step - accuracy: 0.5086 - loss: 1.4130 - val_accuracy: 0.5482 - val_loss: 1.2773 - learning_rate: 0.0010
Restoring model weights from the end of the best epoch: 2.
Epoch 1/3
46/46 ━━━━━━━━━━━━━━━━━━━━ 121s 2s/step - accuracy: 0.5534 - loss: 1.2939 - val_accuracy: 0.5427 - val_loss: 1.2562 - learning_rate: 1.0000e-05
Epoch 2/3
46/46 ━━━━━━━━━━━━━━━━━━━━ 38s 829ms/step - accuracy: 0.5575 - loss: 1.2812 - val_accuracy: 0.5399 - val_loss: 1.2364 - learning_rate: 1.0000e-05
Epoch 3/3
46/46 ━━━━━━━━━━━━━━━━━━━━ 39s 856ms/step - accuracy: 0.5817 - loss: 1.2372 - val_accuracy: 0.5537 - val_loss: 1.2260 - learning_rate: 1.0000e-05
Restoring model weights from the end of the best epoch: 3.


In [42]:
all_results["DenseNet121"] = evaluate_model(
    densenet121_model, test_gen_densenet121, "DenseNet121"
)


3/3 ━━━━━━━━━━━━━━━━━━━━ 30s 9s/step

DenseNet121 — test results:
  Model: DenseNet121
  Accuracy (%): 52.5000
  Precision (%): 49.3266
  Recall (%): 52.5000
  F1-Score (%): 48.4213
  AUC (%): 82.5977
  Inference Time (s): 30.2564
  Params (M): 7.3391

Classification report:
                            precision    recall  f1-score   support

pigmented benign keratosis     0.4737    0.5625    0.5143        16
                  melanoma     0.3333    0.0625    0.1053        16
                     nevus     0.5417    0.8125    0.6500        16
      basal cell carcinoma     0.5294    0.5625    0.5455        16
   squamous cell carcinoma     0.5882    0.6250    0.6061        16

                  accuracy                         0.5250        80
                 macro avg     0.4933    0.5250    0.4842        80
              weighted avg     0.4933    0.5250    0.4842        80

Confusion matrix:
[[ 9  0  0  4  3]
 [ 5  1 10  0  0]
 [ 0  2 13  1  0]
 [ 3  0  0  9  4]
 [ 2  0  1  3 10]]


## ResNet18 (PyTorch, ImageNet-pretrained)

In [43]:
import torch
import torch.nn as nn
from torchvision import models, transforms
from torch.utils.data import Dataset, DataLoader
from PIL import Image

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# ---- IMPROVEMENT: stronger augmentation for the PyTorch models too ----
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomRotation(30),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.RandomAffine(degrees=0, translate=(0.15, 0.15), shear=15, scale=(0.85, 1.15)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])
eval_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

class SkinDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.df = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image = Image.open(row["filename"]).convert("RGB")
        label = SELECTED_CLASSES.index(row["class"])
        if self.transform:
            image = self.transform(image)
        return image, label

train_ds_pt = SkinDataset(train_df, transform=train_transform)
val_ds_pt = SkinDataset(val_df, transform=eval_transform)
test_ds_pt = SkinDataset(test_df, transform=eval_transform)

train_loader_pt = DataLoader(train_ds_pt, batch_size=BATCH_SIZE, shuffle=True)
val_loader_pt = DataLoader(val_ds_pt, batch_size=BATCH_SIZE, shuffle=False)
test_loader_pt = DataLoader(test_ds_pt, batch_size=BATCH_SIZE, shuffle=False)

# ---- IMPROVEMENT: class weights used in the PyTorch loss too (original ResNet18 head-only training skipped this) ----
class_counts_pt = np.bincount(train_labels_split, minlength=NUM_CLASSES)
class_weights_pt = torch.tensor(
    len(train_labels_split) / (NUM_CLASSES * class_counts_pt), dtype=torch.float32
).to(device)
print("PyTorch class weights:", class_weights_pt)


Device: cuda
PyTorch class weights: tensor([0.7864, 0.8291, 1.0147, 0.9641, 2.0014], device='cuda:0')


In [44]:
def train_pytorch_model(model, trainable_params, epochs, lr, weight_decay):
    criterion = nn.CrossEntropyLoss(weight=class_weights_pt, label_smoothing=0.1)
    optimizer = torch.optim.AdamW(trainable_params, lr=lr, weight_decay=weight_decay)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=2)

    best_val_acc, best_state, patience, bad_epochs = 0.0, None, 4, 0

    for epoch in range(epochs):
        model.train()
        for images, labels in train_loader_pt:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

        model.eval()
        val_correct, val_total, val_loss_total = 0, 0, 0.0
        with torch.no_grad():
            for images, labels in val_loader_pt:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                val_loss_total += criterion(outputs, labels).item()
                preds = torch.argmax(outputs, dim=1)
                val_correct += (preds == labels).sum().item()
                val_total += labels.size(0)
        val_acc = val_correct / val_total
        val_loss = val_loss_total / len(val_loader_pt)
        scheduler.step(val_loss)
        print(f"Epoch {epoch+1}/{epochs} — val_loss: {val_loss:.4f}, val_acc: {val_acc:.4f}")

        if val_acc > best_val_acc:
            best_val_acc, best_state, bad_epochs = val_acc, {k: v.clone() for k, v in model.state_dict().items()}, 0
        else:
            bad_epochs += 1
            if bad_epochs >= patience:
                print("Early stopping.")
                break

    if best_state is not None:
        model.load_state_dict(best_state)
    return model


def evaluate_pytorch_model(model, loader, model_name, n_params):
    model.eval()
    y_true, y_pred, y_prob = [], [], []
    start_time = time.time()
    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            outputs = model(images)
            probs = torch.softmax(outputs, dim=1)
            preds = torch.argmax(probs, dim=1)
            y_true.extend(labels.numpy())
            y_pred.extend(preds.cpu().numpy())
            y_prob.extend(probs.cpu().numpy())
    inference_time = time.time() - start_time
    y_true, y_pred, y_prob = np.array(y_true), np.array(y_pred), np.array(y_prob)

    result = {
        "Model": model_name,
        "Accuracy (%)": accuracy_score(y_true, y_pred) * 100,
        "Precision (%)": precision_score(y_true, y_pred, average="macro", zero_division=0) * 100,
        "Recall (%)": recall_score(y_true, y_pred, average="macro", zero_division=0) * 100,
        "F1-Score (%)": f1_score(y_true, y_pred, average="macro", zero_division=0) * 100,
        "AUC (%)": roc_auc_score(y_true, y_prob, multi_class="ovr", average="macro") * 100,
        "Inference Time (s)": inference_time,
        "Params (M)": n_params / 1e6,
    }
    print(f"\n{model_name} — test results:")
    for k, v in result.items():
        print(f"  {k}: {v:.4f}" if isinstance(v, float) else f"  {k}: {v}")
    print(classification_report(y_true, y_pred, target_names=SELECTED_CLASSES, digits=4, zero_division=0))
    print(confusion_matrix(y_true, y_pred))
    return result


In [45]:
resnet18_weights = models.ResNet18_Weights.DEFAULT
resnet18_transfer = models.resnet18(weights=resnet18_weights)
resnet18_transfer.fc = nn.Linear(resnet18_transfer.fc.in_features, NUM_CLASSES)
resnet18_transfer = resnet18_transfer.to(device)

# Phase 1: train the head only
for param in resnet18_transfer.parameters():
    param.requires_grad = False
for param in resnet18_transfer.fc.parameters():
    param.requires_grad = True

resnet18_transfer = train_pytorch_model(
    resnet18_transfer, resnet18_transfer.fc.parameters(),
    epochs=8, lr=1e-3, weight_decay=1e-4,
)

# Phase 2: unfreeze last block (layer4) and fine-tune — original notebook never fine-tuned ResNet18 at all
for param in resnet18_transfer.layer4.parameters():
    param.requires_grad = True
resnet18_transfer = train_pytorch_model(
    resnet18_transfer,
    list(resnet18_transfer.layer4.parameters()) + list(resnet18_transfer.fc.parameters()),
    epochs=8, lr=1e-5, weight_decay=1e-5,
)


Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 164MB/s]


Epoch 1/8 — val_loss: 1.4410, val_acc: 0.3774
Epoch 2/8 — val_loss: 1.3533, val_acc: 0.5069
Epoch 3/8 — val_loss: 1.3530, val_acc: 0.4931
Epoch 4/8 — val_loss: 1.3588, val_acc: 0.5207
Epoch 5/8 — val_loss: 1.2672, val_acc: 0.5813
Epoch 6/8 — val_loss: 1.2789, val_acc: 0.5758
Epoch 7/8 — val_loss: 1.2426, val_acc: 0.5785
Epoch 8/8 — val_loss: 1.2546, val_acc: 0.5785
Epoch 1/8 — val_loss: 1.2215, val_acc: 0.6088
Epoch 2/8 — val_loss: 1.1919, val_acc: 0.5950
Epoch 3/8 — val_loss: 1.1796, val_acc: 0.6226
Epoch 4/8 — val_loss: 1.1650, val_acc: 0.6336
Epoch 5/8 — val_loss: 1.1654, val_acc: 0.6474
Epoch 6/8 — val_loss: 1.1545, val_acc: 0.6501
Epoch 7/8 — val_loss: 1.1305, val_acc: 0.6446
Epoch 8/8 — val_loss: 1.1274, val_acc: 0.6556


In [46]:
n_params_r18 = sum(p.numel() for p in resnet18_transfer.parameters())
all_results["ResNet18"] = evaluate_pytorch_model(resnet18_transfer, test_loader_pt, "ResNet18", n_params_r18)



ResNet18 — test results:
  Model: ResNet18
  Accuracy (%): 58.7500
  Precision (%): 56.8042
  Recall (%): 58.7500
  F1-Score (%): 54.9988
  AUC (%): 84.3164
  Inference Time (s): 4.0286
  Params (M): 11.1791
                            precision    recall  f1-score   support

pigmented benign keratosis     0.6000    0.5625    0.5806        16
                  melanoma     0.4286    0.1875    0.2609        16
                     nevus     0.5926    1.0000    0.7442        16
      basal cell carcinoma     0.6190    0.8125    0.7027        16
   squamous cell carcinoma     0.6000    0.3750    0.4615        16

                  accuracy                         0.5875        80
                 macro avg     0.5680    0.5875    0.5500        80
              weighted avg     0.5680    0.5875    0.5500        80

[[ 9  2  3  1  1]
 [ 6  3  6  0  1]
 [ 0  0 16  0  0]
 [ 0  0  1 13  2]
 [ 0  2  1  7  6]]


## AlexNet (PyTorch, ImageNet-pretrained)

In [49]:
alexnet_weights = models.AlexNet_Weights.DEFAULT
alexnet = models.alexnet(weights=alexnet_weights)
alexnet.classifier[6] = nn.Linear(alexnet.classifier[6].in_features, NUM_CLASSES)
alexnet = alexnet.to(device)

for param in alexnet.parameters():
    param.requires_grad = False
for param in alexnet.classifier.parameters():
    param.requires_grad = True

alexnet = train_pytorch_model(
    alexnet, alexnet.classifier.parameters(),
    epochs=5, lr=1e-4, weight_decay=1e-4,
)

# Fine-tune the last conv block too — original AlexNet only ever trained the classifier head
for param in alexnet.features[-3:].parameters():
    param.requires_grad = True
alexnet = train_pytorch_model(
    alexnet,
    list(alexnet.features[-3:].parameters()) + list(alexnet.classifier.parameters()),
    epochs=4, lr=1e-5, weight_decay=1e-5,
)


Epoch 1/5 — val_loss: 1.3276, val_acc: 0.5455
Epoch 2/5 — val_loss: 1.2842, val_acc: 0.5758
Epoch 3/5 — val_loss: 1.2270, val_acc: 0.6336
Epoch 4/5 — val_loss: 1.1905, val_acc: 0.5868
Epoch 5/5 — val_loss: 1.1285, val_acc: 0.6281
Epoch 1/4 — val_loss: 1.1949, val_acc: 0.6584
Epoch 2/4 — val_loss: 1.1892, val_acc: 0.6694
Epoch 3/4 — val_loss: 1.1633, val_acc: 0.6694
Epoch 4/4 — val_loss: 1.1588, val_acc: 0.6639


In [50]:
n_params_alex = sum(p.numel() for p in alexnet.parameters())
all_results["AlexNet"] = evaluate_pytorch_model(alexnet, test_loader_pt, "AlexNet", n_params_alex)



AlexNet — test results:
  Model: AlexNet
  Accuracy (%): 50.0000
  Precision (%): 59.5326
  Recall (%): 50.0000
  F1-Score (%): 48.5103
  AUC (%): 83.2227
  Inference Time (s): 4.1222
  Params (M): 57.0243
                            precision    recall  f1-score   support

pigmented benign keratosis     0.4545    0.6250    0.5263        16
                  melanoma     0.4444    0.5000    0.4706        16
                     nevus     0.6429    0.5625    0.6000        16
      basal cell carcinoma     0.4348    0.6250    0.5128        16
   squamous cell carcinoma     1.0000    0.1875    0.3158        16

                  accuracy                         0.5000        80
                 macro avg     0.5953    0.5000    0.4851        80
              weighted avg     0.5953    0.5000    0.4851        80

[[10  1  0  5  0]
 [ 5  8  3  0  0]
 [ 3  4  9  0  0]
 [ 2  4  0 10  0]
 [ 2  1  2  8  3]]


## Table 1 — Comparison of Transfer Learning Models
Built directly from `all_results`, which was filled in above as each model was trained and evaluated — nothing here is hard-coded.

In [51]:
from IPython.display import display

table1_df = pd.DataFrame(all_results).T
table1_df = table1_df[["Accuracy (%)", "Precision (%)", "Recall (%)", "F1-Score (%)", "AUC (%)", "Inference Time (s)", "Params (M)"]]
table1_df = table1_df.sort_values("Accuracy (%)", ascending=False).round(2)

display(
    table1_df.style
    .background_gradient(subset=["Accuracy (%)"], cmap="Greens")
    .set_caption("Table 1. Comparison of Transfer Learning Models (sorted by accuracy)")
)
table1_df


,Accuracy (%),Precision (%),Recall (%),F1-Score (%),AUC (%),Inference Time (s),Params (M)
EfficientNetB0,63.750000,64.806536,63.750000,62.831754,90.820312,22.762656,4.417704
ResNet50,61.250000,63.380952,61.250000,60.872278,87.207031,12.129888,24.155525
ResNet18,58.750000,56.804233,58.750000,54.998839,84.316406,4.028562,11.179077
VGG16,52.500000,55.668376,52.500000,47.775115,84.628906,12.173072,14.883141
DenseNet121,52.500000,49.326625,52.500000,48.421280,82.597656,30.256448,7.339077
AlexNet,50.000000,59.532593,50.000000,48.510280,83.222656,4.122181,57.024325
ResNet101,48.750000,46.796388,48.750000,44.829538,81.933594,16.807343,43.225989
VGG19,46.250000,29.229083,46.250000,35.455024,78.359375,4.789737,20.192837


,Accuracy (%),Precision (%),Recall (%),F1-Score (%),AUC (%),Inference Time (s),Params (M)
EfficientNetB0,63.75,64.806536,63.75,62.831754,90.820312,22.762656,4.417704
ResNet50,61.25,63.380952,61.25,60.872278,87.207031,12.129888,24.155525
ResNet18,58.75,56.804233,58.75,54.998839,84.316406,4.028562,11.179077
VGG16,52.5,55.668376,52.5,47.775115,84.628906,12.173072,14.883141
DenseNet121,52.5,49.326625,52.5,48.42128,82.597656,30.256448,7.339077
AlexNet,50.0,59.532593,50.0,48.51028,83.222656,4.122181,57.024325
ResNet101,48.75,46.796388,48.75,44.829538,81.933594,16.807343,43.225989
VGG19,46.25,29.229083,46.25,35.455024,78.359375,4.789737,20.192837


## Table 2 — Classifiers on EfficientNet-B0 Deep Features
Deep features are re-extracted here from the fine-tuned EfficientNet-B0 above (its GlobalAveragePooling output), then 8 classical classifiers are trained on top.

In [53]:
# Rebuild EfficientNet-B0 feature extractor safely

# First, explicitly build/call the model
_ = efficientnetb0_model(
    tf.zeros((1, IMG_SIZE[0], IMG_SIZE[1], 3))
)

# Get the EfficientNet-B0 backbone
efficientnet_base = efficientnetb0_model.layers[0]

# Create feature extractor from the backbone
feature_extractor = keras.Model(
    inputs=efficientnet_base.input,
    outputs=efficientnet_base.output,
)

def extract_features(df):
    datagen = ImageDataGenerator(
        preprocessing_function=None,
        rescale=1.0/255.0
    )

    gen = datagen.flow_from_dataframe(
        df,
        x_col="filename",
        y_col="class",
        target_size=IMG_SIZE,
        batch_size=BATCH_SIZE,
        class_mode="categorical",
        classes=SELECTED_CLASSES,
        shuffle=False,
    )

    features = feature_extractor.predict(gen, verbose=1)
    labels = gen.classes

    return features, labels


X_train_feat, y_train_feat = extract_features(train_df)
X_val_feat, y_val_feat = extract_features(val_df)
X_test_feat, y_test_feat = extract_features(test_df)

# Combine train + validation
X_train_full = np.concatenate([X_train_feat, X_val_feat])
y_train_full = np.concatenate([y_train_feat, y_val_feat])

print("Feature shapes:", X_train_full.shape, X_test_feat.shape)

Found 1451 validated image filenames belonging to 5 classes.
46/46 ━━━━━━━━━━━━━━━━━━━━ 31s 483ms/step
Found 363 validated image filenames belonging to 5 classes.
12/12 ━━━━━━━━━━━━━━━━━━━━ 5s 376ms/step
Found 80 validated image filenames belonging to 5 classes.
3/3 ━━━━━━━━━━━━━━━━━━━━ 8s 4s/step
Feature shapes: (1814, 7, 7, 1280) (80, 7, 7, 1280)


In [55]:
# Rebuild EfficientNet-B0 feature extractor with Global Average Pooling

_ = efficientnetb0_model(
    tf.zeros((1, IMG_SIZE[0], IMG_SIZE[1], 3))
)

efficientnet_base = efficientnetb0_model.layers[0]

feature_extractor = keras.Sequential([
    efficientnet_base,
    layers.GlobalAveragePooling2D()
])

def extract_features(df):
    datagen = ImageDataGenerator(
        preprocessing_function=None,
        rescale=1.0/255.0
    )

    gen = datagen.flow_from_dataframe(
        df,
        x_col="filename",
        y_col="class",
        target_size=IMG_SIZE,
        batch_size=BATCH_SIZE,
        class_mode="categorical",
        classes=SELECTED_CLASSES,
        shuffle=False,
    )

    features = feature_extractor.predict(gen, verbose=1)
    labels = gen.classes

    return features, labels


X_train_feat, y_train_feat = extract_features(train_df)
X_val_feat, y_val_feat = extract_features(val_df)
X_test_feat, y_test_feat = extract_features(test_df)

# Combine train + validation
X_train_full = np.concatenate([X_train_feat, X_val_feat])
y_train_full = np.concatenate([y_train_feat, y_val_feat])

print("Feature shapes:")
print("Train:", X_train_full.shape)
print("Test:", X_test_feat.shape)


Found 1451 validated image filenames belonging to 5 classes.
46/46 ━━━━━━━━━━━━━━━━━━━━ 35s 606ms/step
Found 363 validated image filenames belonging to 5 classes.
12/12 ━━━━━━━━━━━━━━━━━━━━ 8s 642ms/step
Found 80 validated image filenames belonging to 5 classes.
3/3 ━━━━━━━━━━━━━━━━━━━━ 7s 3s/step
Feature shapes:
Train: (1814, 1280)
Test: (80, 1280)


In [56]:
# ============================================================
# TABLE 2: COMPARISON OF DIFFERENT CLASSIFIERS
# EfficientNet-B0 Deep Features
# ============================================================

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)
from xgboost import XGBClassifier
import numpy as np
import time

# ------------------------------------------------------------
# 1. Standardize features
# ------------------------------------------------------------

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train_full)
X_test_scaled = scaler.transform(X_test_feat)

print("Scaled feature shapes:")
print("Train:", X_train_scaled.shape)
print("Test :", X_test_scaled.shape)


# ------------------------------------------------------------
# 2. Define classifiers
# ------------------------------------------------------------

classifiers = {

    "Logistic Regression": (
        LogisticRegression(
            max_iter=2000,
            class_weight="balanced"
        ),
        True
    ),

    "Decision Tree": (
        DecisionTreeClassifier(
            class_weight="balanced",
            random_state=SEED
        ),
        False
    ),

    "Random Forest": (
        RandomForestClassifier(
            n_estimators=300,
            class_weight="balanced",
            random_state=SEED,
            n_jobs=-1
        ),
        False
    ),

    "K-Nearest Neighbors": (
        KNeighborsClassifier(
            n_neighbors=7
        ),
        True
    ),

    "Linear SVM": (
        SVC(
            kernel="linear",
            probability=True,
            class_weight="balanced",
            random_state=SEED
        ),
        True
    ),

    "RBF-SVM": (
        SVC(
            kernel="rbf",
            probability=True,
            class_weight="balanced",
            random_state=SEED
        ),
        True
    ),

    "XGBoost": (
        XGBClassifier(
            n_estimators=300,
            eval_metric="mlogloss",
            random_state=SEED,
            n_jobs=-1
        ),
        False
    )
}


# ------------------------------------------------------------
# 3. Train and evaluate all classifiers
# ------------------------------------------------------------

table2_results = {}

for name, (clf, needs_scaling) in classifiers.items():

    print("\n" + "=" * 60)
    print("Training:", name)
    print("=" * 60)

    # Select scaled or original features
    if needs_scaling:
        Xtr = X_train_scaled
        Xte = X_test_scaled
    else:
        Xtr = X_train_full
        Xte = X_test_feat

    # -------------------------
    # Training time
    # -------------------------
    start = time.time()

    clf.fit(Xtr, y_train_full)

    train_time = time.time() - start

    # -------------------------
    # Prediction time
    # -------------------------
    start = time.time()

    y_pred = clf.predict(Xte)

    inference_time = time.time() - start

    # -------------------------
    # Probability for AUC
    # -------------------------
    if hasattr(clf, "predict_proba"):
        y_prob = clf.predict_proba(Xte)
    else:
        y_prob = None

    # -------------------------
    # Calculate metrics
    # -------------------------

    accuracy = accuracy_score(
        y_test_feat,
        y_pred
    ) * 100

    precision = precision_score(
        y_test_feat,
        y_pred,
        average="macro",
        zero_division=0
    ) * 100

    recall = recall_score(
        y_test_feat,
        y_pred,
        average="macro",
        zero_division=0
    ) * 100

    f1 = f1_score(
        y_test_feat,
        y_pred,
        average="macro",
        zero_division=0
    ) * 100

    if y_prob is not None:
        auc = roc_auc_score(
            y_test_feat,
            y_prob,
            multi_class="ovr",
            average="macro"
        ) * 100
    else:
        auc = np.nan

    # -------------------------
    # Save results
    # -------------------------

    table2_results[name] = {
        "Accuracy (%)": accuracy,
        "Precision (%)": precision,
        "Recall (%)": recall,
        "F1-Score (%)": f1,
        "AUC (%)": auc,
        "Train Time (s)": train_time,
        "Inference Time (s)": inference_time
    }

    # -------------------------
    # Display result
    # -------------------------

    print(
        f"{name} done — "
        f"Accuracy: {accuracy:.2f}% | "
        f"Precision: {precision:.2f}% | "
        f"Recall: {recall:.2f}% | "
        f"F1: {f1:.2f}% | "
        f"AUC: {auc:.2f}%"
    )


# ============================================================
# 4. Create Table 2
# ============================================================

import pandas as pd

table2_df = pd.DataFrame(table2_results).T

print("\n\n")
print("=" * 100)
print("TABLE 2: COMPARISON OF DIFFERENT CLASSIFIERS")
print("=" * 100)

display(
    table2_df[
        [
            "Accuracy (%)",
            "Precision (%)",
            "Recall (%)",
            "F1-Score (%)",
            "AUC (%)"
        ]
    ].round(2)
)

Scaled feature shapes:
Train: (1814, 1280)
Test : (80, 1280)

Training: Logistic Regression
Logistic Regression done — Accuracy: 45.00% | Precision: 46.42% | Recall: 45.00% | F1: 43.99% | AUC: 69.69%

Training: Decision Tree
Decision Tree done — Accuracy: 32.50% | Precision: 30.52% | Recall: 32.50% | F1: 30.28% | AUC: 57.81%

Training: Random Forest
Random Forest done — Accuracy: 38.75% | Precision: 34.52% | Recall: 38.75% | F1: 34.67% | AUC: 73.59%

Training: K-Nearest Neighbors
K-Nearest Neighbors done — Accuracy: 32.50% | Precision: 32.17% | Recall: 32.50% | F1: 27.75% | AUC: 67.27%

Training: Linear SVM
Linear SVM done — Accuracy: 46.25% | Precision: 46.94% | Recall: 46.25% | F1: 45.68% | AUC: 74.92%

Training: RBF-SVM
RBF-SVM done — Accuracy: 43.75% | Precision: 41.16% | Recall: 43.75% | F1: 40.64% | AUC: 79.02%

Training: XGBoost
XGBoost done — Accuracy: 43.75% | Precision: 43.45% | Recall: 43.75% | F1: 40.45% | AUC: 72.62%



TABLE 2: COMPARISON OF DIFFERENT CLASSIFIERS


,Accuracy (%),Precision (%),Recall (%),F1-Score (%),AUC (%)
Logistic Regression,45.00,46.42,45.00,43.99,69.69
Decision Tree,32.50,30.52,32.50,30.28,57.81
Random Forest,38.75,34.52,38.75,34.67,73.59
K-Nearest Neighbors,32.50,32.17,32.50,27.75,67.27
Linear SVM,46.25,46.94,46.25,45.68,74.92
RBF-SVM,43.75,41.16,43.75,40.64,79.02
XGBoost,43.75,43.45,43.75,40.45,72.62


In [57]:
table2_df = pd.DataFrame(table2_results).T.round(2)
table2_df = table2_df.sort_values("Accuracy (%)", ascending=False)

display(
    table2_df.style
    .background_gradient(subset=["Accuracy (%)"], cmap="Blues")
    .set_caption("Table 2. Comparison of Classifiers on EfficientNet-B0 Deep Features")
)
table2_df


,Accuracy (%),Precision (%),Recall (%),F1-Score (%),AUC (%),Train Time (s),Inference Time (s)
Linear SVM,46.250000,46.940000,46.250000,45.680000,74.920000,34.190000,0.040000
Logistic Regression,45.000000,46.420000,45.000000,43.990000,69.690000,22.000000,0.000000
RBF-SVM,43.750000,41.160000,43.750000,40.640000,79.020000,11.350000,0.120000
XGBoost,43.750000,43.450000,43.750000,40.450000,72.620000,118.910000,0.010000
Random Forest,38.750000,34.520000,38.750000,34.670000,73.590000,16.670000,0.090000
Decision Tree,32.500000,30.520000,32.500000,30.280000,57.810000,3.920000,0.000000
K-Nearest Neighbors,32.500000,32.170000,32.500000,27.750000,67.270000,0.000000,0.220000


,Accuracy (%),Precision (%),Recall (%),F1-Score (%),AUC (%),Train Time (s),Inference Time (s)
Linear SVM,46.25,46.94,46.25,45.68,74.92,34.19,0.04
Logistic Regression,45.00,46.42,45.00,43.99,69.69,22.00,0.00
RBF-SVM,43.75,41.16,43.75,40.64,79.02,11.35,0.12
XGBoost,43.75,43.45,43.75,40.45,72.62,118.91,0.01
Random Forest,38.75,34.52,38.75,34.67,73.59,16.67,0.09
Decision Tree,32.50,30.52,32.50,30.28,57.81,3.92,0.00
K-Nearest Neighbors,32.50,32.17,32.50,27.75,67.27,0.00,0.22


## Table 3 — Computational Efficiency Comparison
Combines parameter counts (already captured in `all_results` while training), FLOPs, and measured inference time per image.

In [58]:
def count_flops_keras(model, input_shape=(1, 224, 224, 3)):
    forward_pass = tf.function(lambda x: model(x))
    concrete = forward_pass.get_concrete_function(tf.TensorSpec(input_shape, tf.float32))
    graph_info = tf.compat.v1.profiler.profile(
        concrete.graph,
        options=tf.compat.v1.profiler.ProfileOptionBuilder.float_operation()
    )
    return graph_info.total_float_ops / 2  # MACs

keras_models_dict = {
    "EfficientNetB0": efficientnetb0_model,
    "VGG16": vgg16_model,
    "VGG19": vgg19_model,
    "ResNet50": resnet50_model,
    "ResNet101": resnet101_model,
    "DenseNet121": densenet121_model,
}

efficiency_rows = {}
for name, model in keras_models_dict.items():
    try:
        flops = count_flops_keras(model)
    except Exception as e:
        flops = np.nan
        print(f"Could not compute FLOPs for {name}: {e}")
    efficiency_rows[name] = {
        "Params (M)": all_results[name]["Params (M)"],
        "FLOPs (G)": flops / 1e9 if not np.isnan(flops) else np.nan,
        "Inference Time (s)": all_results[name]["Inference Time (s)"],
        "Accuracy (%)": all_results[name]["Accuracy (%)"],
    }


Instructions for updating:
This API was designed for TensorFlow v1. See https://www.tensorflow.org/guide/migrate for instructions on how to migrate your code to TensorFlow v2.


In [59]:
from thop import profile
import torch

def count_flops_pytorch(model, input_size=(1, 3, 224, 224)):
    dummy = torch.randn(input_size).to(device)
    flops, params = profile(model, inputs=(dummy,), verbose=False)
    return flops

for name, model in [("ResNet18", resnet18_transfer), ("AlexNet", alexnet)]:
    try:
        flops = count_flops_pytorch(model)
    except Exception as e:
        flops = np.nan
        print(f"Could not compute FLOPs for {name}: {e}")
    efficiency_rows[name] = {
        "Params (M)": all_results[name]["Params (M)"],
        "FLOPs (G)": flops / 1e9 if not np.isnan(flops) else np.nan,
        "Inference Time (s)": all_results[name]["Inference Time (s)"],
        "Accuracy (%)": all_results[name]["Accuracy (%)"],
    }


In [60]:
table3_df = pd.DataFrame(efficiency_rows).T.round(3)
table3_df = table3_df.sort_values("Accuracy (%)", ascending=False)

display(
    table3_df.style
    .background_gradient(subset=["Accuracy (%)"], cmap="Oranges")
    .set_caption("Table 3. Computational Efficiency Comparison")
)
table3_df


,Params (M),FLOPs (G),Inference Time (s),Accuracy (%)
EfficientNetB0,4.418000,0.401000,22.763000,63.750000
ResNet50,24.156000,3.876000,12.130000,61.250000
ResNet18,11.179000,1.824000,4.029000,58.750000
VGG16,14.883000,15.357000,12.173000,52.500000
DenseNet121,7.339000,2.851000,30.256000,52.500000
AlexNet,57.024000,0.710000,4.122000,50.000000
ResNet101,43.226000,7.598000,16.807000,48.750000
VGG19,20.193000,19.519000,4.790000,46.250000


,Params (M),FLOPs (G),Inference Time (s),Accuracy (%)
EfficientNetB0,4.418,0.401,22.763,63.75
ResNet50,24.156,3.876,12.130,61.25
ResNet18,11.179,1.824,4.029,58.75
VGG16,14.883,15.357,12.173,52.50
DenseNet121,7.339,2.851,30.256,52.50
AlexNet,57.024,0.710,4.122,50.00
ResNet101,43.226,7.598,16.807,48.75
VGG19,20.193,19.519,4.790,46.25
